# 02 MMTIS Loading and Preparation

In this notebook, we load the actual train operation data from the MMTIS dataset.

The goal is to calculate train delays and prepare clean datasets for later visualizations.

## 1. Import libraries

First, we import the libraries that we need for data loading and basic data preparation.

In [58]:
import pandas as pd
from pathlib import Path

## 2. Define data paths

We define the path to the raw MMTIS data and to the folder where we will save processed files.

In [59]:
raw_path = Path("../data/raw/mmtis")
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

## 3. Find all train operation files

The MMTIS folder contains weekly CSV files.  
We use the files named `zugfahrten.csv`, because they contain all train trips, not only delayed trains.

In [60]:
trip_files = sorted(
    raw_path.rglob("*_zugfahrten.csv")
)

len(trip_files)

27

In [61]:
trip_files[:5]

[PosixPath('../data/raw/mmtis/2025/2025-12-01_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-08_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-15_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-22_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-29_mmtis_zugfahrten.csv')]

## 4. Load all train operation files

We load all weekly train operation files and combine them into one dataset.

In [62]:
dfs = []

for file in trip_files:
    df = pd.read_csv(file)
    dfs.append(df)

trips = pd.concat(dfs, ignore_index=True)

In [63]:
trips.shape

(1053324, 9)

In [64]:
trips.head()

,zugnummer,betriebstag,abfahrtzeit_soll,abfahrtzeit_ist,start_betriebsstelle,ankunftzeit_soll,ankunftzeit_ist,ziel_betriebsstelle,fahrzeit_delta
0,20,2025-11-17,2025-11-17T17:15:00+0100,2025-11-17T17:19:13+0100,MLX,NaN,2025-11-17T19:53:37+0100,PAG,NaN
1,21,2025-11-17,2025-11-17T10:21:30+0100,2025-11-17T10:34:31+0100,PAG,2025-11-17T12:44:45+0100,2025-11-17T12:49:46+0100,MLX,-00:08:00
2,22,2025-11-17,2025-11-17T15:15:00+0100,2025-11-17T15:15:42+0100,MLX,NaN,2025-11-17T17:41:51+0100,PAG,NaN
3,23,2025-11-17,2025-11-17T12:22:54+0100,2025-11-17T12:56:25+0100,PAG,2025-11-17T14:44:45+0100,2025-11-17T15:18:47+0100,MLX,00:00:31
4,26,2025-11-17,2025-11-17T11:15:00+0100,2025-11-17T11:19:33+0100,MLX,NaN,2025-11-17T13:37:02+0100,PAG,NaN


## 5. Inspect the dataset

Now we check the columns, data types, and missing values.

In [65]:
trips.columns.tolist()

['zugnummer',
 'betriebstag',
 'abfahrtzeit_soll',
 'abfahrtzeit_ist',
 'start_betriebsstelle',
 'ankunftzeit_soll',
 'ankunftzeit_ist',
 'ziel_betriebsstelle',
 'fahrzeit_delta']

In [66]:
trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1053324 entries, 0 to 1053323
Data columns (total 9 columns):
 #   Column                Non-Null Count    Dtype 
---  ------                --------------    ----- 
 0   zugnummer             1053324 non-null  int64 
 1   betriebstag           1053324 non-null  object
 2   abfahrtzeit_soll      1049472 non-null  object
 3   abfahrtzeit_ist       1047032 non-null  object
 4   start_betriebsstelle  1053324 non-null  object
 5   ankunftzeit_soll      1044231 non-null  object
 6   ankunftzeit_ist       1046983 non-null  object
 7   ziel_betriebsstelle   1053324 non-null  object
 8   fahrzeit_delta        1034256 non-null  object
dtypes: int64(1), object(8)
memory usage: 72.3+ MB


In [67]:
trips.isna().sum()

zugnummer                   0
betriebstag                 0
abfahrtzeit_soll         3852
abfahrtzeit_ist          6292
start_betriebsstelle        0
ankunftzeit_soll         9093
ankunftzeit_ist          6341
ziel_betriebsstelle         0
fahrzeit_delta          19068
dtype: int64

## 6. Convert time columns

The time columns are loaded as text.  
We convert them to datetime format so that we can calculate delays.

In [71]:
time_columns = [
    "abfahrtzeit_soll",
    "abfahrtzeit_ist",
    "ankunftzeit_soll",
    "ankunftzeit_ist"
]

for col in time_columns:
    trips[col] = pd.to_datetime(
        trips[col],
        errors="coerce"
    )

In [72]:
trips[time_columns].dtypes

abfahrtzeit_soll    datetime64[ns, UTC+01:00]
abfahrtzeit_ist     datetime64[ns, UTC+01:00]
ankunftzeit_soll    datetime64[ns, UTC+01:00]
ankunftzeit_ist     datetime64[ns, UTC+01:00]
dtype: object

## 7. Calculate delays

We calculate departure delay and arrival delay in minutes.

A positive value means that the train was late.  
A negative value means that the train was earlier than scheduled.

In [73]:
trips["departure_delay_min"] = (
    trips["abfahrtzeit_ist"] - trips["abfahrtzeit_soll"]
).dt.total_seconds() / 60

In [74]:
trips["arrival_delay_min"] = (
    trips["ankunftzeit_ist"] - trips["ankunftzeit_soll"]
).dt.total_seconds() / 60

In [75]:
trips[[
    "zugnummer",
    "betriebstag",
    "departure_delay_min",
    "arrival_delay_min"
]].head()

,zugnummer,betriebstag,departure_delay_min,arrival_delay_min
0,20,2025-11-17,4.216667,NaN
1,21,2025-11-17,13.016667,5.016667
2,22,2025-11-17,0.700000,NaN
3,23,2025-11-17,33.516667,34.033333
4,26,2025-11-17,4.550000,NaN


## 8. Create time features

We create simple time features from the scheduled departure time.

These features will help us analyze delays by hour, weekday, month, and time of day.

In [76]:
trips["departure_hour"] = trips["abfahrtzeit_soll"].dt.hour
trips["weekday"] = trips["abfahrtzeit_soll"].dt.day_name()
trips["month"] = trips["abfahrtzeit_soll"].dt.month_name()

In [77]:
trips[[
    "abfahrtzeit_soll",
    "departure_hour",
    "weekday",
    "month"
]].head()

,abfahrtzeit_soll,departure_hour,weekday,month
0,2025-11-17 17:15:00+01:00,17.0,Monday,November
1,2025-11-17 10:21:30+01:00,10.0,Monday,November
2,2025-11-17 15:15:00+01:00,15.0,Monday,November
3,2025-11-17 12:22:54+01:00,12.0,Monday,November
4,2025-11-17 11:15:00+01:00,11.0,Monday,November


## 9. Create time-of-day categories

We group the departure hour into simple categories: morning peak, midday, evening peak, and night.

In [78]:
def get_time_of_day(hour):
    if pd.isna(hour):
        return "unknown"
    elif 6 <= hour < 9:
        return "morning_peak"
    elif 9 <= hour < 16:
        return "midday"
    elif 16 <= hour < 19:
        return "evening_peak"
    else:
        return "night"

In [79]:
trips["time_of_day"] = trips["departure_hour"].apply(get_time_of_day)

In [80]:
trips["time_of_day"].value_counts()

time_of_day
unknown         318037
midday          258982
night           225640
morning_peak    126115
evening_peak    124550
Name: count, dtype: int64

## 10. Prepare delay analysis dataset

Some records do not have scheduled or actual arrival times.  
For delay analysis, we keep only records where arrival delay can be calculated.

In [81]:
delay_analysis = trips.dropna(
    subset=["arrival_delay_min"]
).copy()

In [82]:
delay_analysis.shape

(726251, 15)

In [83]:
delay_analysis["arrival_delay_min"].describe()

count    726251.000000
mean          1.993230
std           7.176926
min        -560.483333
25%          -0.183333
50%           0.450000
75%           2.066667
max         782.733333
Name: arrival_delay_min, dtype: float64

## 11. Handle extreme delay values

Some delay values are very large or very negative.  
For visualizations, we create a capped delay column.

This keeps the main patterns visible and reduces the effect of extreme outliers.

In [85]:
delay_analysis["arrival_delay_capped"] = (
    delay_analysis["arrival_delay_min"]
    .clip(lower=-30, upper=120)
)

In [86]:
delay_analysis[[
    "arrival_delay_min",
    "arrival_delay_capped"
]].describe()

,arrival_delay_min,arrival_delay_capped
count,726251.000000,726251.000000
mean,1.993230,1.994114
std,7.176926,6.378355
min,-560.483333,-30.000000
25%,-0.183333,-0.183333
50%,0.450000,0.450000
75%,2.066667,2.066667
max,782.733333,120.000000


## 12. Check punctuality

We define a train as on time if it arrives no more than 5 minutes late.

In [87]:
delay_analysis["on_time"] = (
    delay_analysis["arrival_delay_min"] <= 5
)

In [88]:
on_time_rate = delay_analysis["on_time"].mean()

on_time_rate

np.float64(0.898782927665504)

## 13. Create summary table by hour

This table shows the average arrival delay for each departure hour.

In [89]:
hour_summary = (
    delay_analysis
    .groupby("departure_hour", as_index=False)
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

hour_summary

,departure_hour,mean_delay,median_delay,number_of_trips
0,0.0,1.608176,0.050000,9620
1,1.0,1.086015,-0.100000,2343
2,2.0,1.459279,0.083333,1383
3,3.0,2.771694,0.166667,2997
4,4.0,1.254605,0.266667,22593
5,5.0,1.738155,0.416667,41973
6,6.0,2.124502,0.600000,47695
7,7.0,2.288440,0.550000,40174
8,8.0,2.186856,0.441667,35898
9,9.0,1.908482,0.350000,35134


## 14. Create summary table by weekday

This table shows the average arrival delay for each weekday.

In [90]:
weekday_summary = (
    delay_analysis
    .groupby("weekday", as_index=False)
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

weekday_summary

,weekday,mean_delay,median_delay,number_of_trips
0,Friday,2.383747,0.600000,109559
1,Monday,2.093075,0.550000,108853
2,Saturday,1.350812,0.200000,93061
3,Sunday,1.303620,0.133333,84943
4,Thursday,2.188361,0.583333,108209
5,Tuesday,2.266212,0.566667,108937
6,Wednesday,2.120418,0.566667,110091


## 15. Create summary table by station pair

This table shows average delay between start and destination operating points.

In [91]:
station_pair_summary = (
    delay_analysis
    .groupby(
        ["start_betriebsstelle", "ziel_betriebsstelle"],
        as_index=False
    )
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

In [92]:
station_pair_summary.sort_values(
    "mean_delay",
    ascending=False
).head(20)

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
1400,JB,LN,120.0,120.0,1
644,FB,GZS,120.0,120.0,1
1159,HE,HW H1,120.0,120.0,1
2988,RO H2,STTS12,120.0,120.0,1
1289,HP U1,MTH,120.0,120.0,1
2830,PG,PG G,120.0,120.0,1
2796,PAG,PW,120.0,120.0,2
2791,PAG,LZW,120.0,120.0,1
2790,PAG,HEZ,120.0,120.0,1
1445,K S11,I,120.0,120.0,1


## 16. Save processed datasets

We save the prepared trip-level data and the summary tables.  
These files will be used later for visualizations and the dashboard.

In [93]:
delay_analysis.to_csv(
    processed_path / "mmtis_delay_analysis.csv",
    index=False
)

hour_summary.to_csv(
    processed_path / "mmtis_hour_summary.csv",
    index=False
)

weekday_summary.to_csv(
    processed_path / "mmtis_weekday_summary.csv",
    index=False
)

station_pair_summary.to_csv(
    processed_path / "mmtis_station_pair_summary.csv",
    index=False
)

In [94]:
print("Saved files:")
print(processed_path / "mmtis_delay_analysis.csv")
print(processed_path / "mmtis_hour_summary.csv")
print(processed_path / "mmtis_weekday_summary.csv")
print(processed_path / "mmtis_station_pair_summary.csv")

Saved files:
../data/processed/mmtis_delay_analysis.csv
../data/processed/mmtis_hour_summary.csv
../data/processed/mmtis_weekday_summary.csv
../data/processed/mmtis_station_pair_summary.csv


## 17. Basic data quality checks

Before moving to visualizations, we check whether the processed delay dataset looks reasonable.

We check:
- number of rows
- missing values
- delay distribution
- very early and very late trains
- number of unique station pairs

In [95]:
print("Rows in delay_analysis:", delay_analysis.shape[0])
print("Columns in delay_analysis:", delay_analysis.shape[1])

delay_analysis[
    [
        "arrival_delay_min",
        "arrival_delay_capped",
        "departure_delay_min",
        "departure_hour",
        "weekday",
        "month",
        "time_of_day"
    ]
].describe()

Rows in delay_analysis: 726251
Columns in delay_analysis: 17


,arrival_delay_min,arrival_delay_capped,departure_delay_min,departure_hour
count,726251.000000,726251.000000,723633.000000,723653.000000
mean,1.993230,1.994114,1.388932,12.795459
std,7.176926,6.378355,26.229808,5.687617
min,-560.483333,-30.000000,-15840.000000,0.000000
25%,-0.183333,-0.183333,0.033333,8.000000
50%,0.450000,0.450000,0.500000,13.000000
75%,2.066667,2.066667,1.350000,17.000000
max,782.733333,120.000000,419.883333,23.000000


## 18. Missing values in the final dataset

We check missing values after filtering the data for valid arrival timestamps.
This helps us understand which columns are safe to use for analysis and visualization.

In [96]:
delay_analysis.isna().sum().sort_values(ascending=False).head(20)

fahrzeit_delta          2618
departure_delay_min     2618
abfahrtzeit_soll        2598
month                   2598
weekday                 2598
departure_hour          2598
abfahrtzeit_ist           20
arrival_delay_min          0
arrival_delay_capped       0
time_of_day                0
zugnummer                  0
betriebstag                0
ziel_betriebsstelle        0
ankunftzeit_ist            0
ankunftzeit_soll           0
start_betriebsstelle       0
on_time                    0
dtype: int64

## 19. Check extreme delay values

Some delay values are very large or very negative.  
These observations may be caused by cancellations, data errors, or special operational cases.

For visualizations, we use a capped delay metric, but we still inspect the original values.

In [97]:
delay_analysis.sort_values("arrival_delay_min").head(10)[
    [
        "zugnummer",
        "start_betriebsstelle",
        "ziel_betriebsstelle",
        "ankunftzeit_soll",
        "ankunftzeit_ist",
        "arrival_delay_min"
    ]
]

,zugnummer,start_betriebsstelle,ziel_betriebsstelle,ankunftzeit_soll,ankunftzeit_ist,arrival_delay_min
259102,2155,SH,SH,2026-01-03 05:07:00+01:00,2026-01-02 19:46:31+01:00,-560.483333
155167,3440,AU,AU,2025-12-15 04:45:00+01:00,2025-12-14 22:07:06+01:00,-397.900000
372503,5015,W,W,2026-01-23 07:17:00+01:00,2026-01-23 01:15:53+01:00,-361.116667
58022,3902,SL,SL,2025-11-27 06:11:00+01:00,2025-11-27 00:32:49+01:00,-338.183333
174291,5625,BG H1,BG H1,2025-12-18 06:09:00+01:00,2025-12-18 00:33:09+01:00,-335.850000
286593,3780,KG,KG,2026-01-08 05:27:00+01:00,2026-01-07 23:51:42+01:00,-335.300000
166226,2107,GM,GM,2025-12-17 05:09:00+01:00,2025-12-17 00:07:08+01:00,-301.866667
494256,21094,TU,TU,2026-02-13 04:59:00+01:00,2026-02-13 00:00:26+01:00,-298.566667
46220,3400,AT,AT,2025-11-25 04:38:00+01:00,2025-11-24 23:46:00+01:00,-292.000000
539012,26014,BL,BL,2026-02-21 05:27:00+01:00,2026-02-21 00:38:50+01:00,-288.166667


In [98]:
delay_analysis.sort_values("arrival_delay_min", ascending=False).head(10)[
    [
        "zugnummer",
        "start_betriebsstelle",
        "ziel_betriebsstelle",
        "ankunftzeit_soll",
        "ankunftzeit_ist",
        "arrival_delay_min"
    ]
]

,zugnummer,start_betriebsstelle,ziel_betriebsstelle,ankunftzeit_soll,ankunftzeit_ist,arrival_delay_min
561542,25862,GOTA1,WON,2026-02-25 17:25:49+01:00,2026-02-26 06:28:33+01:00,782.733333
301853,294,TBV,SBM,2026-01-11 06:55:47+01:00,2026-01-11 14:24:52+01:00,449.083333
525252,4630,GU Z2,LIEH1,2026-02-19 21:14:01+01:00,2026-02-20 03:33:55+01:00,379.900000
301827,234,VBO,MLX,2026-01-11 09:03:54+01:00,2026-01-11 15:21:52+01:00,377.966667
312977,1031,BPA,OF,2026-01-13 10:19:25+01:00,2026-01-13 15:25:19+01:00,305.900000
431981,462,HE,OF,2026-02-03 23:18:26+01:00,2026-02-04 04:21:33+01:00,303.116667
330449,253,BPA,OF,2026-01-16 16:47:50+01:00,2026-01-16 21:49:42+01:00,301.866667
471962,663,WO S12,MLX,2026-02-10 23:47:42+01:00,2026-02-11 04:38:19+01:00,290.616667
316609,8207,STP,STPG,2026-01-13 07:02:30+01:00,2026-01-13 11:36:30+01:00,274.000000
525259,4637,LIEH1,UZ S11,2026-02-19 21:44:06+01:00,2026-02-20 02:14:13+01:00,270.116667


## 20. Create heatmap summary table

This table shows average delay by weekday and hour.  
It will be used later for the time heatmap in the dashboard.

In [99]:
heatmap_summary = (
    delay_analysis
    .groupby(["weekday", "departure_hour"])
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_min", "median"),
        number_of_trips=("arrival_delay_min", "count")
    )
    .reset_index()
)

heatmap_summary.head()

,weekday,departure_hour,mean_delay,median_delay,number_of_trips
0,Friday,0.0,1.451728,0.050000,1283
1,Friday,1.0,0.979819,-0.275000,166
2,Friday,2.0,2.961111,0.200000,48
3,Friday,3.0,3.125575,0.283333,391
4,Friday,4.0,1.514890,0.316667,3742


In [100]:
heatmap_summary.to_csv(
    processed_path / "mmtis_heatmap_summary.csv",
    index=False
)

## 21. Create top delayed station pairs table

For the dashboard, we prepare a ranked table of station pairs with the highest average delay.

To make the result more reliable, we only keep station pairs with at least 30 observations.

In [101]:
top_station_pairs = (
    station_pair_summary
    .query("number_of_trips >= 30")
    .sort_values("mean_delay", ascending=False)
    .head(30)
)

top_station_pairs

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
2784,PA,MLX,43.920531,30.800000,69
2202,MLX,ND G,33.543229,14.991667,32
2137,MKI,I W1,23.970000,24.733333,45
2793,PAG,MLX,23.777624,13.950000,1048
1154,HE,BC,23.306631,14.333333,93
1327,HW Z3,HE,21.472917,1.975000,112
329,BPA,VBO,20.867228,12.983333,297
2182,MLX,JB,19.913864,14.900000,113
1510,KLSS22,BC,19.352899,14.766667,92
1336,HW Z3,NIFG,18.458889,0.833333,30


In [102]:
top_station_pairs.to_csv(
    processed_path / "mmtis_top_station_pairs.csv",
    index=False
)